# DSA 8301 — Statistical Inference for Big Data
## Kenya Housing Survey 2023/24 — Data Loading, Understanding & Exploration

**Student:** [Your Name] | **Reg No:** [Your Reg No]  
**Course:** DSA 8301 — Statistical Inference for Big Data  
**Lecturer:** Prof. Jacob Ong'ala  
**Institution:** Strathmore University  
**Date:** June 2026  

---

### Dataset Source
Kenya National Bureau of Statistics (KNBS) — *Kenya Housing Survey 2023/24*  
Portal: https://statistics.knbs.or.ke/nada/index.php/catalog/184/get-microdata

---

> **Scope of this notebook:** Data loading → variable inventory → preprocessing → descriptive statistics → graphical EDA → distributional assessment.  
> Parametric and non-parametric inference follow in a separate notebook.


---
## 0. Environment Setup


In [1]:
# ── 0.1  Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


Mounted at /content/drive
Drive mounted.


In [2]:
# ── 0.2  Install dependencies (first run only) ───────────────────────────
!pip install -q pyreadstat polars pyarrow
print('Dependencies ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.6 MB/s eta 0:00:00
Dependencies ready.


In [3]:
# ── 0.3  Core imports ────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, probplot, norm as spnorm
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.fontsize': 9,
})

TEAL   = '#00695C'; RED    = '#B71C1C'; AMBER  = '#E65100'
BLUE   = '#1565C0'; PURPLE = '#6A1B9A'; GRAY   = '#546E7A'
DARK   = '#2C2C2A'; GREEN  = '#2E7D32'

print('All imports loaded.')


All imports loaded.


In [4]:
# ── 0.4  Paths and county map ────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ    = DRIVE / 'data' / 'parquet'
RAW   = DRIVE / 'data' / 'raw'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301'
for p in [FIGS, TABS]: p.mkdir(parents=True, exist_ok=True)

COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}
print(f'Paths ready.  FIGS={FIGS}  TABS={TABS}')


Paths ready.  FIGS=/content/drive/MyDrive/KHS_Dissertation/outputs/figures/dsa8301  TABS=/content/drive/MyDrive/KHS_Dissertation/outputs/tables/dsa8301


---
## 1. Dataset Description

### 1.1 Source & Background

The **Kenya Housing Survey (KHS) 2023/24** is a nationally representative household survey conducted by KNBS. It covers **21,347 households** across all **47 counties**, collecting data on housing conditions, tenure, infrastructure, household finances, and demographic composition.

The survey ships as six Stata (.dta) files, converted here to Parquet for efficiency:

| File key | Unit of observation | Core content |
|----------|---------------------|--------------|
| `household` | Household (spine) | Finances, tenure, utilities, infrastructure — 392 columns |
| `individual` | Person | Demographics, education, employment |
| `dwelling` | Dwelling unit | Wall/roof/floor materials, rooms, floor area |
| `land_parcels` | Land parcel | Tenure system, title documents, eviction risk |
| `county` | County (47 rows) | Physical planning, infrastructure indicators |
| `mortgage` | Mortgage record | Demand, products, county-level coverage |


In [5]:
# ── 1.2  Load all parquet files ─────────────────────────────────────────
FILES = {
    'household'   : 'Household_Information_Data.parquet',
    'individual'  : 'Individual_Data.parquet',
    'dwelling'    : 'Dwelling_Units_Data.parquet',
    'land_parcels': 'Land_Parcels_Data.parquet',
    'county'      : 'County_Physical_Planning_Data.parquet',
    'mortgage'    : 'Housing_Mortgage_Data.parquet',


    'loan'         : 'Housing_Loans_Data.parquet',
    # New — previously unused
    'nema'         : 'NEMA_Data_Set.parquet',
    'water_svc'    : 'Water_Services_Providers_Data.parquet',
    'real_estate'  : 'Real_Estate_Dataset.parquet',
    'financiers'   : 'Housing_Financiers_Data.parquet',
    'institutional': 'KHS_Institutional_Data.parquet',
    'project_info' : 'Project Information.parquet',
    'housing_types': 'Type of Housing Units.parquet',



}

dfs = {}
print(f'  {"File":<15} {"Rows":>8} {"Cols":>6}')
print('  ' + '-'*32)
for key, fname in FILES.items():
    path = PQ / fname
    if not path.exists():
        print(f'  {key:<15} NOT FOUND')
        continue
    df = pd.read_parquet(path)
    dfs[key] = df
    print(f'  {key:<15} {df.shape[0]:>8,} {df.shape[1]:>6}')

hh  = dfs.get('household')
ind = dfs.get('individual')
dw  = dfs.get('dwelling')
print(f'\nPrimary frame: household  =>  {hh.shape[0]:,} rows x {hh.shape[1]:,} cols')


  File                Rows   Cols
  --------------------------------
  household         21,347    392
  individual        80,889     97
  dwelling          25,116     25
  land_parcels      11,136     34
  county                47    116
  mortgage           1,644     13
  loan                 946     10
  nema                  48     45
  water_svc            153     96
  real_estate        7,236    300
  financiers           351     63
  institutional        348    194
  project_info          71    211
  housing_types        131     17

Primary frame: household  =>  21,347 rows x 392 cols


In [6]:
# ── 1.3  Variable Registry ──────────────────────────────────────────────
# A reference map of analysis-relevant variables across all loaded files.
# type: continuous | ordinal | binary | categorical

VARIABLE_REGISTRY = {

    # ── IDENTIFIERS & WEIGHTS ────────────────────────────────────────────
    'interview__key' : {'label': 'Household unique interview key',                   'type': 'id',          'file': 'household'},
    'a01'            : {'label': 'County code (1–47)',                               'type': 'categorical', 'file': 'household'},
    'countycode'     : {'label': 'County code string-padded (01–47)',                'type': 'categorical', 'file': 'household'},
    'a07_1'          : {'label': 'Urban/Rural stratum (1=Urban, 2=Rural)',           'type': 'binary',      'file': 'household'},
    'serial'         : {'label': 'KNBS household serial number',                     'type': 'id',          'file': 'household'},
    'hhweight'       : {'label': 'Household survey weight',                          'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — WATER & SANITATION ───────────────────────────────────
    'c01_1'          : {'label': 'Main drinking water source (1=piped HH, 4=borehole, 10=river)', 'type': 'ordinal',     'file': 'household'},
    'c01_2'          : {'label': 'Water collection method (1=piped, 5=fetched)',     'type': 'ordinal',     'file': 'household'},
    'c01_3'          : {'label': 'Water treated before drinking (0=No, 1=Yes)',      'type': 'binary',      'file': 'household'},
    'c01_4'          : {'label': 'Time to water source (minutes, one way)',          'type': 'continuous',  'file': 'household'},
    'c02_1'          : {'label': 'Secondary drinking water source',                  'type': 'ordinal',     'file': 'household'},
    'c04'            : {'label': 'Main toilet facility type (1=flush, 7=pit, 8=none)', 'type': 'ordinal',   'file': 'household'},
    'c05'            : {'label': 'Handwashing facility available (0=No, 1=Yes)',     'type': 'binary',      'file': 'household'},
    'c07'            : {'label': 'Handwashing materials present (4=soap+water)',     'type': 'ordinal',     'file': 'household'},
    'c14_1'          : {'label': 'Monthly water expenditure (KES)',                  'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — ENERGY ───────────────────────────────────────────────
    'c10'            : {'label': 'Main lighting source (1=grid electricity, 4=solar, 5=kerosene)', 'type': 'ordinal', 'file': 'household'},
    'c10_2'          : {'label': 'Hours of electricity supply per day',              'type': 'continuous',  'file': 'household'},
    'c10_4'          : {'label': 'Electricity connection type (0=prepaid, 1=postpaid)', 'type': 'binary',   'file': 'household'},
    'c11'            : {'label': 'Main cooking fuel (7=firewood, 9=charcoal, 11=LPG)', 'type': 'ordinal',  'file': 'household'},
    'c12'            : {'label': 'Primary cooking stove type (1=3-stone, 6=improved, 10=gas)', 'type': 'ordinal', 'file': 'household'},
    'c14_2'          : {'label': 'Monthly electricity expenditure (KES)',            'type': 'continuous',  'file': 'household'},
    'c14_3'          : {'label': 'Monthly other energy expenditure (KES)',           'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — ASSETS ───────────────────────────────────────────────
    'c13__1'         : {'label': 'Owns radio (0=No, 1=Yes)',                         'type': 'binary',      'file': 'household'},
    'c13__2'         : {'label': 'Owns mobile phone (0=No, 1=Yes)',                  'type': 'binary',      'file': 'household'},
    'c13__3'         : {'label': 'Owns television (0=No, 1=Yes)',                    'type': 'binary',      'file': 'household'},
    'c13__4'         : {'label': 'Owns computer/laptop (0=No, 1=Yes)',               'type': 'binary',      'file': 'household'},
    'c13__5'         : {'label': 'Owns motorcycle (0=No, 1=Yes)',                    'type': 'binary',      'file': 'household'},
    'c13__6'         : {'label': 'Owns motor vehicle (0=No, 1=Yes)',                 'type': 'binary',      'file': 'household'},
    'c13__7'         : {'label': 'Owns refrigerator (0=No, 1=Yes)',                  'type': 'binary',      'file': 'household'},
    'internet'       : {'label': 'Household has internet access (1=Yes)',            'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — INCOME & EXPENDITURE ─────────────────────────────────
    'g01a'           : {'label': 'Monthly food expenditure (KES)',                   'type': 'continuous',  'file': 'household'},
    'g01b'           : {'label': 'Monthly clothing expenditure (KES)',               'type': 'continuous',  'file': 'household'},
    'g01c'           : {'label': 'Monthly education expenditure (KES)',              'type': 'continuous',  'file': 'household'},
    'g01d'           : {'label': 'Monthly health expenditure (KES)',                 'type': 'continuous',  'file': 'household'},
    'g01e'           : {'label': 'Monthly transport expenditure (KES)',              'type': 'continuous',  'file': 'household'},
    'g01f'           : {'label': 'Monthly communication expenditure (KES)',          'type': 'continuous',  'file': 'household'},
    'g01g'           : {'label': 'Monthly recreation expenditure (KES)',             'type': 'continuous',  'file': 'household'},
    'g01h'           : {'label': 'Monthly housing cost expenditure (KES)',           'type': 'continuous',  'file': 'household'},
    'g01i'           : {'label': 'Monthly energy expenditure (KES)',                 'type': 'continuous',  'file': 'household'},
    'g01j'           : {'label': 'Monthly other expenditure (KES)',                  'type': 'continuous',  'file': 'household'},
    'g01k'           : {'label': 'Monthly remittances sent (KES)',                   'type': 'continuous',  'file': 'household'},
    'g02'            : {'label': 'Household pays rent (1=Yes, 2=No)',                'type': 'binary',      'file': 'household'},
    'g02_1'          : {'label': 'Monthly rent paid (KES)',                          'type': 'continuous',  'file': 'household'},
    'g03'            : {'label': 'Housing tenure type (1=owner, 2=tenant, 3=other)', 'type': 'categorical', 'file': 'household'},
    'g04'            : {'label': 'Owns other property (0=No, 1=Yes)',                'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — HOUSING PROBLEMS & ASPIRATIONS ───────────────────────
    'g05__1'         : {'label': 'Problem: overcrowding (0=No, 1=Yes)',              'type': 'binary',      'file': 'household'},
    'g05__2'         : {'label': 'Problem: poor water supply (0=No, 1=Yes)',         'type': 'binary',      'file': 'household'},
    'g05__3'         : {'label': 'Problem: poor sanitation (0=No, 1=Yes)',           'type': 'binary',      'file': 'household'},
    'g05__4'         : {'label': 'Problem: poor drainage (0=No, 1=Yes)',             'type': 'binary',      'file': 'household'},
    'g05__5'         : {'label': 'Problem: poor road access (0=No, 1=Yes)',          'type': 'binary',      'file': 'household'},
    'g05__6'         : {'label': 'Problem: insecurity (0=No, 1=Yes)',                'type': 'binary',      'file': 'household'},
    'g05__7'         : {'label': 'Problem: high rent/housing cost (0=No, 1=Yes)',    'type': 'binary',      'file': 'household'},
    'g05__8'         : {'label': 'Problem: poor structural condition (0=No, 1=Yes)', 'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — HOUSING PERCEPTION (h01–h11) ─────────────────────────
    'h01'            : {'label': 'Perceived structural quality (1=Good, 2=Fair, 3=Poor)',    'type': 'ordinal', 'file': 'household'},
    'h02'            : {'label': 'Perceived roof quality (1=Good, 2=Fair, 3=Poor)',          'type': 'ordinal', 'file': 'household'},
    'h03'            : {'label': 'Perceived wall quality (1=Good, 2=Fair, 3=Poor)',          'type': 'ordinal', 'file': 'household'},
    'h04'            : {'label': 'Perceived floor quality (1=Good, 2=Fair, 3=Poor)',         'type': 'ordinal', 'file': 'household'},
    'h05'            : {'label': 'Perceived ventilation adequacy (1=Good, 2=Fair, 3=Poor)',  'type': 'ordinal', 'file': 'household'},
    'h06'            : {'label': 'Perceived natural lighting (1=Good, 2=Fair, 3=Poor)',      'type': 'ordinal', 'file': 'household'},
    'h07'            : {'label': 'Perceived water supply adequacy (1=Good, 2=Fair, 3=Poor)', 'type': 'ordinal', 'file': 'household'},
    'h08'            : {'label': 'Perceived sanitation adequacy (1=Good, 2=Fair, 3=Poor)',   'type': 'ordinal', 'file': 'household'},
    'h09'            : {'label': 'Perceived waste disposal (1=Good, 2=Fair, 3=Poor)',        'type': 'ordinal', 'file': 'household'},
    'h10'            : {'label': 'Perceived neighbourhood security (1=Good, 2=Fair, 3=Poor)','type': 'ordinal', 'file': 'household'},
    'h11'            : {'label': 'Overall housing satisfaction (1=Good, 2=Fair, 3=Poor)',    'type': 'ordinal', 'file': 'household'},

    # ── HOUSEHOLD — TENURE & MOBILITY (j-module) ─────────────────────────
    'j04_1'          : {'label': 'Current tenure arrangement (1=owner, 0=other)',    'type': 'binary',      'file': 'household'},
    'j05'            : {'label': 'Has formal title/ownership document (0=No, 1=Yes)','type': 'binary',      'file': 'household'},
    'j09'            : {'label': 'Housing cost is a financial burden (0=No, 1=Yes)', 'type': 'binary',      'file': 'household'},
    'j10'            : {'label': 'Ever missed rent/mortgage payment (0=No, 1=Yes)',  'type': 'binary',      'file': 'household'},
    'j11'            : {'label': 'At risk of eviction in next 12 months (0=No, 1=Yes)', 'type': 'binary',   'file': 'household'},
    'j12_1'          : {'label': 'Years in current dwelling (1=<1yr, coded year otherwise)', 'type': 'ordinal', 'file': 'household'},
    'j13'            : {'label': 'Satisfied with current tenure (0=No, 1=Yes)',      'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — RENTAL MODULE (k-module, renters only ~32%) ──────────
    'k02'            : {'label': 'Written tenancy agreement exists (1=Yes, 2=No)',   'type': 'binary',      'file': 'household'},
    'k05'            : {'label': 'Monthly rent (KES)',                               'type': 'continuous',  'file': 'household'},
    'k09'            : {'label': 'Lease type (1=monthly, 2=annual)',                 'type': 'categorical', 'file': 'household'},
    'k21'            : {'label': 'Rent arrears status (1=in arrears, others)',       'type': 'ordinal',     'file': 'household'},
    'min_rent'       : {'label': 'Minimum rent in PSU — local market floor (KES)',   'type': 'continuous',  'file': 'household'},

    # ── HOUSEHOLD — OWNED DWELLING MODULE (l-module) ─────────────────────
    'l07'            : {'label': 'Year dwelling was built',                          'type': 'continuous',  'file': 'household'},
    'l13'            : {'label': 'Monthly mortgage/housing loan repayment (KES)',    'type': 'continuous',  'file': 'household'},
    'l14'            : {'label': 'Estimated market value of dwelling (KES)',         'type': 'continuous',  'file': 'household'},
    'l15'            : {'label': 'Imputed monthly housing cost/rent equivalent (KES)', 'type': 'continuous','file': 'household'},
    'l19'            : {'label': 'Plot/land size (decimal: 12=1 acre)',              'type': 'continuous',  'file': 'household'},
    'l21'            : {'label': 'Major renovation done (0=No, 1=Yes)',              'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — ENVIRONMENT & HAZARDS (e-module) ─────────────────────
    'e01'            : {'label': 'Solid waste disposal method (0=collected, 9=open dump)', 'type': 'ordinal', 'file': 'household'},
    'e05'            : {'label': 'Proximity to waste dump/quarry (0=No, 1=Yes)',     'type': 'binary',      'file': 'household'},
    'e06'            : {'label': 'Flood exposure (0=none, 1=severe, 2=mild)',        'type': 'ordinal',     'file': 'household'},
    'e07'            : {'label': 'Mudslide/erosion exposure (0=none, 1=severe, 2=mild)', 'type': 'ordinal', 'file': 'household'},
    'e08'            : {'label': 'Terrain/slope type (1=flat, 2=gentle, 3=hilly, 4=steep)', 'type': 'ordinal', 'file': 'household'},

    # ── HOUSEHOLD — LAND OWNERSHIP (i00) ─────────────────────────────────
    'i00'            : {'label': 'Household owns land (0=No, 1=Yes)',                'type': 'binary',      'file': 'household'},

    # ── HOUSEHOLD — DERIVED / COMPUTED ───────────────────────────────────
    'prop_util'      : {'label': 'Utilities as proportion of income (ratio)',        'type': 'continuous',  'file': 'household'},
    'med_prop'       : {'label': 'Median utility-to-income ratio, county level',     'type': 'continuous',  'file': 'household'},
    'utilities'      : {'label': 'Household pays for utilities (0=No, 1=Yes)',       'type': 'binary',      'file': 'household'},
    'ctymin_ut'      : {'label': 'County-level minimum utility cost (KES)',          'type': 'continuous',  'file': 'household'},
    'med_brms'       : {'label': 'Median bedrooms in county (rooms)',                'type': 'continuous',  'file': 'household'},
    'sf'             : {'label': 'Slum/informal settlement flag (1=slum, 2=non-slum)', 'type': 'binary',    'file': 'household'},
    'pln'            : {'label': 'Planning status of settlement (coded)',            'type': 'categorical', 'file': 'household'},

    # ── DWELLING FILE ─────────────────────────────────────────────────────
    'd01'            : {'label': 'Dwelling tenure type (1=owner-occupied, 2=rented, 3=other)', 'type': 'categorical', 'file': 'dwelling'},
    'd03'            : {'label': 'Dwelling type (1=conventional house, 4=flat, 7=traditional)', 'type': 'categorical', 'file': 'dwelling'},
    'd05'            : {'label': 'Located in approved building (1=Yes, 0=No)',       'type': 'binary',      'file': 'dwelling'},
    'd06'            : {'label': 'Building has planning approval (1=Yes, 0=No)',     'type': 'binary',      'file': 'dwelling'},
    'd07'            : {'label': 'Dwelling in hazard-prone area (1=Yes, 0=No)',      'type': 'binary',      'file': 'dwelling'},
    'd08'            : {'label': 'Outer wall material (1=stone/brick, 3=timber, 5=mud/earth)', 'type': 'ordinal', 'file': 'dwelling'},
    'd09'            : {'label': 'Roof material (1=iron sheets, 2=tiles, 3=grass/thatch)', 'type': 'ordinal', 'file': 'dwelling'},
    'd10'            : {'label': 'Floor material (1=cement, 2=tiles, 3=earth)',      'type': 'ordinal',     'file': 'dwelling'},
    'd11'            : {'label': 'Number of rooms in dwelling',                      'type': 'continuous',  'file': 'dwelling'},
    'd11_1'          : {'label': 'Floor area of dwelling (sq m)',                    'type': 'continuous',  'file': 'dwelling'},
    'd11_2'          : {'label': 'Number of bedrooms',                              'type': 'continuous',  'file': 'dwelling'},
    'd12'            : {'label': 'Number of rooms used for sleeping',                'type': 'continuous',  'file': 'dwelling'},

    # ── INDIVIDUAL FILE ───────────────────────────────────────────────────
    'b04'            : {'label': 'Sex of individual (1=Male, 2=Female)',             'type': 'binary',      'file': 'individual'},
    'b05_years'      : {'label': 'Age in completed years',                           'type': 'continuous',  'file': 'individual'},
    'b07'            : {'label': 'Marital status (1=never married, 2=married, 3=divorced, 4=widowed)', 'type': 'categorical', 'file': 'individual'},
    'b10'            : {'label': 'Currently attending school (0=No, 1=Yes)',         'type': 'binary',      'file': 'individual'},
    'b11'            : {'label': 'Literacy status (0=illiterate, 1=literate)',       'type': 'binary',      'file': 'individual'},
    'ken_edu_isced11': {'label': 'Highest education level (ISCED-11: 3=primary, 6=secondary, 16=tertiary)', 'type': 'ordinal', 'file': 'individual'},
    'any_disability' : {'label': 'Any functional disability (0=No, 1=Yes)',          'type': 'binary',      'file': 'individual'},
    'resid'          : {'label': 'Residence type (1=Urban, 2=Rural)',                'type': 'binary',      'file': 'individual'},
    'hhsize'         : {'label': 'Total persons in household',                       'type': 'continuous',  'file': 'individual'},
    'age_dep'        : {'label': 'Age dependency status (0=child <15, 15=working age, 65=elderly)', 'type': 'ordinal', 'file': 'individual'},
    'wap'            : {'label': 'Working-age population flag (1=WAP, NaN=not)',     'type': 'binary',      'file': 'individual'},
    'inw'            : {'label': 'Individual survey weight',                         'type': 'continuous',  'file': 'individual'},

    # ── LAND PARCELS FILE ─────────────────────────────────────────────────
    'i01_3'          : {'label': 'Land ownership type (1=freehold, 2=leasehold, 3=customary)', 'type': 'categorical', 'file': 'land_parcels'},
    'i05'            : {'label': 'Land has title deed (1=Yes, 3=no, 14=other)',      'type': 'ordinal',     'file': 'land_parcels'},
    'i06'            : {'label': 'Land use type (1=residential, 12=agricultural, 14=mixed)', 'type': 'categorical', 'file': 'land_parcels'},
    'i08'            : {'label': 'Land dispute in last 5 years (0=No, 1=Yes)',       'type': 'binary',      'file': 'land_parcels'},
    'i10'            : {'label': 'Land registered (0=No, 1=Yes)',                   'type': 'binary',      'file': 'land_parcels'},
    'i12'            : {'label': 'Land used as loan collateral (0=No, 1=Yes)',       'type': 'binary',      'file': 'land_parcels'},

}

# ── Summary ──────────────────────────────────────────────────────────────
_reg_df = pd.DataFrame(VARIABLE_REGISTRY).T
print(f'  Total variables registered: {len(VARIABLE_REGISTRY)}')
print()
print(_reg_df.groupby(['file', 'type']).size().rename('count').to_string())

  Total variables registered: 125

file          type       
dwelling      binary          3
              categorical     2
              continuous      4
              ordinal         3
household     binary         34
              categorical     5
              continuous     29
              id              2
              ordinal        25
individual    binary          6
              categorical     1
              continuous      3
              ordinal         2
land_parcels  binary          3
              categorical     2
              ordinal         1


In [7]:
# ── 1.4  Join feasibility ────────────────────────────────────────────────
# For every non-household file, test whether interview__key links back
# to the household spine.  Results inform the merge strategy in 1.7.

SPINE_KEY   = 'interview__key'
hh_keys     = set(hh[SPINE_KEY].dropna())
hh_counties = set(hh['a01'].dropna().astype(int))

print("JOIN FEASIBILITY MATRIX")
print("=" * 70)
print(f"Household spine : {len(hh_keys):,} unique interview__key values\n")
print(f"  {'File':<18} {'Rows':>7}  {'HH-level join':>22}  {'County col'}")
print("  " + "─" * 68)

for key, df in dfs.items():
    if key == 'household':
        continue

    n_rows = f"{df.shape[0]:,}"

    # household-level match
    hh_match = ''
    if SPINE_KEY in df.columns:
        file_keys  = set(df[SPINE_KEY].dropna())
        overlap    = len(file_keys & hh_keys)
        pct        = overlap / len(hh_keys) * 100
        hh_match   = f"{overlap:,} / {len(hh_keys):,}  ({pct:.1f}%)"
    else:
        hh_match   = "no interview__key"

    # county bridge column
    cty_col = next(
        (c for c in df.columns if c.lower() in
         ('a01', 'cg00', 'nm00', 'county_name', 'county_code', 'countycode')),
        '—'
    )

    print(f"  {key:<18} {n_rows:>7}  {hh_match:>22}  {cty_col}")

print()
print("  Conclusion")
print("  ─" * 35)
print("  100% match  → direct left join on interview__key")
print("  45.5% match → left join (NaN = household owns no land — see 1.5)")
print("  0% match    → aggregate to county then join via a01")



JOIN FEASIBILITY MATRIX
Household spine : 21,347 unique interview__key values

  File                  Rows           HH-level join  County col
  ────────────────────────────────────────────────────────────────────
  individual          80,889  21,346 / 21,347  (100.0%)  a01
  dwelling            25,116  21,346 / 21,347  (100.0%)  a01
  land_parcels        11,136  9,707 / 21,347  (45.5%)  —
  county                  47      0 / 21,347  (0.0%)  cg00
  mortgage             1,644      0 / 21,347  (0.0%)  county_name
  loan                   946      0 / 21,347  (0.0%)  county_name
  nema                    48      0 / 21,347  (0.0%)  nm00
  water_svc              153      0 / 21,347  (0.0%)  county_name
  real_estate          7,236       no interview__key  county_name
  financiers             351      0 / 21,347  (0.0%)  county_name
  institutional          348      0 / 21,347  (0.0%)  county_name
  project_info            71      0 / 21,347  (0.0%)  —
  housing_types          131      0 

In [8]:


# ── 1.5  Land parcels — structural coverage check ────────────────────────
# 45.5% HH coverage: hypothesis is that parcel records exist iff i00 == 1.

print("\n" + "=" * 70)
print("LAND_PARCELS — is the 45.5% gap explained by i00 (land ownership)?")
print("=" * 70)

lp      = dfs['land_parcels'].copy()
lp_keys = set(lp[SPINE_KEY].dropna())

hh_sub = hh[['interview__key', 'i00']].copy()
hh_sub['has_parcel'] = hh_sub['interview__key'].isin(lp_keys).astype(int)

print(f"\n  Parcel records          : {len(lp):,}")
print(f"  HHs with  parcel record : {len(lp_keys & hh_keys):,}"
      f"  ({len(lp_keys & hh_keys)/len(hh_keys)*100:.1f}%)")
print(f"  HHs without parcel      : {len(hh_keys - lp_keys):,}"
      f"  ({len(hh_keys - lp_keys)/len(hh_keys)*100:.1f}%)")

print(f"\n  i00 × parcel presence:")
print(f"  {'i00':<6} {'Has parcel':>12} {'No parcel':>12} {'% with parcel':>15}")
print("  " + "─" * 48)
for val in sorted(hh_sub['i00'].dropna().unique()):
    sub = hh_sub[hh_sub['i00'] == val]
    has = sub['has_parcel'].sum()
    no  = len(sub) - has
    pct = has / len(sub) * 100
    print(f"  {val:<6.0f} {has:>12,} {no:>12,} {pct:>14.1f}%")

sub_nan = hh_sub[hh_sub['i00'].isna()]
if len(sub_nan):
    has = sub_nan['has_parcel'].sum()
    no  = len(sub_nan) - has
    pct = has / len(sub_nan) * 100 if len(sub_nan) else 0
    print(f"  {'NaN':<6} {has:>12,} {no:>12,} {pct:>14.1f}%")

pph = lp.groupby(SPINE_KEY).size().value_counts().sort_index().to_dict()
print(f"\n  Parcels per household : {pph}")
print("\n  ✓ Structurally complete: parcel record exists iff i00 = 1.")
print("    Left join is correct — NaN means no land owned, not missing data.")



LAND_PARCELS — is the 45.5% gap explained by i00 (land ownership)?

  Parcel records          : 11,136
  HHs with  parcel record : 9,707  (45.5%)
  HHs without parcel      : 11,640  (54.5%)

  i00 × parcel presence:
  i00      Has parcel    No parcel   % with parcel
  ────────────────────────────────────────────────
  0                 0       11,639            0.0%
  1             9,707            0          100.0%
  NaN               0            1            0.0%

  Parcels per household : {1: 8520, 2: 1004, 3: 147, 4: 30, 5: 7, 6: 2}

  ✓ Structurally complete: parcel record exists iff i00 = 1.
    Left join is correct — NaN means no land owned, not missing data.


In [9]:


# ── 1.6  County-level aggregates ──────────────────────────────────────────
# Files with 0% HH-level match are aggregated to 47 county summaries,
# then joined to the master frame via a01 (county code integer).

import unicodedata, re, numpy as np

# County name → integer code lookup (covers messy strings in the data)
def _norm(s):
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r"['\-]", '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

NAME_TO_CODE = {_norm(v): k for k, v in COUNTY_MAP.items()}
NAME_TO_CODE.update({
    'nairobi city'  : 47, 'nairobi'         : 47,
    'muranga'       : 21, 'taita taveta'     : 6,
    'tharaka nithi' : 13, 'elgeyo marakwet'  : 28,
    'uasin gishu'   : 27, 'homa bay'         : 43,
    'homabay'       : 43,
})

def to_county_code(series):
    return series.map(lambda x: NAME_TO_CODE.get(_norm(str(x)), pd.NA))

# ── COUNTY ───────────────────────────────────────────────────────────────
county_df = dfs['county'].copy()
county_df['a01'] = (county_df['cg00'].astype(str).str.strip()
                                      .str.lstrip('0')
                                      .replace('', pd.NA)
                                      .astype(float)
                                      .astype('Int64'))
county_agg = county_df[['a01', 'cg1a', 'cg1b', 'cg3', 'cg11', 'cg12']].rename(columns={
    'cg1a' : 'cty_housing_stock',
    'cg1b' : 'cty_housing_backlog',
    'cg3'  : 'cty_planning_staff',
    'cg11' : 'cty_has_housing_policy',
    'cg12' : 'cty_building_approval_system',
})

# ── NEMA ─────────────────────────────────────────────────────────────────
nema_df = dfs['nema'].copy()
nema_df['a01'] = nema_df['nm00'].astype('Int64')
nema_agg = nema_df[['a01', 'nema1a', 'nema1b', 'nema6']].rename(columns={
    'nema1a' : 'nema_eia_applications',
    'nema1b' : 'nema_eia_approvals',
    'nema6'  : 'nema_processing_days',
})

# ── WATER_SVC ────────────────────────────────────────────────────────────
# county_name stores integer county codes in this file
wsvc_df = dfs['water_svc'].copy()
wsvc_df['a01'] = pd.to_numeric(wsvc_df['county_name'], errors='coerce').astype('Int64')
wsvc_agg = (
    wsvc_df[['a01', 'wssp1a', 'wssp1b', 'wssp7', 'wssp12']]
    .rename(columns={
        'wssp1a'  : 'wsvc_water_connections',
        'wssp1b'  : 'wsvc_sewer_connections',
        'wssp7'   : 'wsvc_water_tariff',
        'wssp12'  : 'wsvc_service_quality',
    })
    .groupby('a01', as_index=False).agg(
        wsvc_water_connections = ('wsvc_water_connections', 'sum'),
        wsvc_sewer_connections = ('wsvc_sewer_connections', 'sum'),
        wsvc_water_tariff      = ('wsvc_water_tariff',      'mean'),
        wsvc_service_quality   = ('wsvc_service_quality',   'mean'),
        wsvc_n_providers       = ('wsvc_water_tariff',      'count'),
    )
)

# ── MORTGAGE ─────────────────────────────────────────────────────────────
mort_df = dfs['mortgage'].copy()
mort_df['a01'] = to_county_code(mort_df['county_name'])
mort_agg = (
    mort_df[['a01', 'se6b', 'se8a', 'se9b']]
    .rename(columns={
        'se6b' : 'mort_interest_rate',
        'se8a' : 'mort_ltv_ratio',
        'se9b' : 'mort_avg_term_years',
    })
    .groupby('a01', as_index=False).agg(
        mort_interest_rate  = ('mort_interest_rate',  'mean'),
        mort_ltv_ratio      = ('mort_ltv_ratio',       'mean'),
        mort_avg_term_years = ('mort_avg_term_years',  'mean'),
        mort_n_providers    = ('mort_interest_rate',   'count'),
    )
)

# ── LOAN ─────────────────────────────────────────────────────────────────
loan_df = dfs['loan'].copy()
loan_df['a01'] = to_county_code(loan_df['county_name'])
loan_agg = (
    loan_df[['a01', 'se3c', 'se3d']]
    .rename(columns={
        'se3c' : 'loan_avg_size',
        'se3d' : 'loan_outstanding',
    })
    .groupby('a01', as_index=False).agg(
        loan_avg_size    = ('loan_avg_size',    'mean'),
        loan_outstanding = ('loan_outstanding', 'mean'),
        loan_n_providers = ('loan_avg_size',    'count'),
    )
)

# ── FINANCIERS ────────────────────────────────────────────────────────────
fin_df = dfs['financiers'].copy()
fin_df['a01'] = to_county_code(fin_df['county_name'])
fin_agg = (
    fin_df[['a01', 'se4a', 'se7']]
    .rename(columns={
        'se4a' : 'fin_portfolio',
        'se7'  : 'fin_avg_tenure_months',
    })
    .groupby('a01', as_index=False).agg(
        fin_portfolio          = ('fin_portfolio',          'sum'),
        fin_avg_tenure_months  = ('fin_avg_tenure_months',  'mean'),
        fin_n_financiers       = ('fin_portfolio',          'count'),
    )
)

print("County aggregates built:")
for name, df_ in [
    ('county_agg', county_agg), ('nema_agg',  nema_agg),
    ('wsvc_agg',   wsvc_agg),   ('mort_agg',  mort_agg),
    ('loan_agg',   loan_agg),   ('fin_agg',   fin_agg),
]:
    print(f"  {name:<14} {df_.shape[0]:>3} rows × {df_.shape[1]:>2} cols  "
          f"| a01 coverage: {df_['a01'].notna().sum()} / 47 counties")



County aggregates built:
  county_agg      47 rows ×  6 cols  | a01 coverage: 47 / 47 counties
  nema_agg        48 rows ×  4 cols  | a01 coverage: 48 / 47 counties
  wsvc_agg        45 rows ×  6 cols  | a01 coverage: 45 / 47 counties
  mort_agg        38 rows ×  5 cols  | a01 coverage: 38 / 47 counties
  loan_agg        38 rows ×  4 cols  | a01 coverage: 38 / 47 counties
  fin_agg         38 rows ×  4 cols  | a01 coverage: 38 / 47 counties


In [10]:
# ── 1.7  Individual → household aggregates ───────────────────────────────

ind = dfs['individual'].copy()

ind_agg = (
    ind.groupby('interview__key', as_index=False).agg(
        hh_size         = ('b02_length',      'first'),
        n_female        = ('b04',             lambda x: (x == 2).sum()),
        n_children      = ('age_dep',         lambda x: (x == 0).sum()),
        n_elderly       = ('age_dep',         lambda x: (x == 65).sum()),
        n_working_age   = ('age_dep',         lambda x: (x == 15).sum()),
        hhh_sex         = ('hhh_sex',         'first'),
        any_disability  = ('any_disability',  'max'),
        max_edu_isced   = ('ken_edu_isced11', 'max'),
        mean_age        = ('age_cur',         'mean'),
    )
)

ind_agg['dependency_ratio'] = np.where(
    ind_agg['n_working_age'] > 0,
    (ind_agg['n_children'] + ind_agg['n_elderly']) / ind_agg['n_working_age'],
    np.nan,
)

print(f"Individual aggregates : {ind_agg.shape[0]:,} rows × {ind_agg.shape[1]} cols")



Individual aggregates : 21,347 rows × 11 cols


In [11]:
# ── 1.8  Dwelling → household aggregates ─────────────────────────────────
# FIX: filter to spine keys BEFORE groupby to prevent row inflation.
# The raw dwelling file has 8 interview__key values absent from the spine;
# keeping them produces 21,354 groupby keys and inflates the merge to 22,406.

dw = dfs['dwelling'].copy()
hh_keys = set(hh['interview__key'].dropna())

dw_spine = dw[dw['interview__key'].isin(hh_keys)]
print(f"Dwelling rows after spine filter : {len(dw_spine):,}  "
      f"(dropped {len(dw) - len(dw_spine)} stray keys)")

dw_agg = (
    dw_spine.sort_values('interview__key')
      .groupby('interview__key', as_index=False)
      .agg(
          dw_type               = ('d03',  'first'),
          dw_wall_material      = ('d08',  'first'),
          dw_roof_material      = ('d09',  'first'),
          dw_floor_material     = ('d10',  'first'),
          dw_n_rooms            = ('d11',  'first'),
          dw_floor_area_m2      = ('d11_1','first'),
          dw_n_bedrooms         = ('d11_2','first'),
          dw_approved           = ('d05',  'first'),
          dw_planning_ok        = ('d06',  'first'),
          dw_hazard_zone        = ('d07',  'first'),
          dw_n_units_enumerated = ('d03',  'count'),
      )
)

assert len(dw_agg) <= len(hh), \
    f"dw_agg still larger than spine: {len(dw_agg):,}"
print(f"Dwelling aggregates   : {dw_agg.shape[0]:,} rows × {dw_agg.shape[1]} cols")



Dwelling rows after spine filter : 25,108  (dropped 8 stray keys)
Dwelling aggregates   : 21,346 rows × 12 cols


In [12]:
# ── 1.9  Land parcels → household aggregates ─────────────────────────────
# Same precaution applied: filter lp to spine keys first.
# (lp has 9,710 unique keys — 3 are outside the spine per join feasibility.)

lp = dfs['land_parcels'].copy()
lp_spine = lp[lp['interview__key'].isin(hh_keys)]
print(f"Land parcel rows after spine filter : {len(lp_spine):,}  "
      f"(dropped {len(lp) - len(lp_spine)} stray keys)")

lp_agg = (
    lp_spine.groupby('interview__key', as_index=False).agg(
        lp_n_parcels       = ('land_parcels__id', 'count'),
        lp_primary_tenure  = ('i01_3',  'first'),
        lp_has_title       = ('i05',    lambda x: int((x == 1).any())),
        lp_primary_use     = ('i06',    'first'),
        lp_any_dispute     = ('i08',    'max'),
        lp_any_registered  = ('i10',    'max'),
        lp_any_collateral  = ('i12',    'max'),
    )
)

print(f"Land parcel aggregates: {lp_agg.shape[0]:,} rows × {lp_agg.shape[1]} cols")


Land parcel rows after spine filter : 11,133  (dropped 3 stray keys)
Land parcel aggregates: 9,707 rows × 8 cols


In [13]:
# ── 1.9b  Fix nema_agg — deduplicate duplicate county code ────────────────
# nema_agg has 48 rows but only 47 counties.  Collapse on a01 by mean
# so the county join never produces a many-to-one expansion.

nema_agg = (
    nema_agg.groupby('a01', as_index=False).agg(
        nema_eia_applications = ('nema_eia_applications', 'sum'),
        nema_eia_approvals    = ('nema_eia_approvals',    'sum'),
        nema_processing_days  = ('nema_processing_days',  'mean'),
    )
)
assert len(nema_agg) <= 47, \
    f"nema_agg still has {len(nema_agg)} rows after dedup"
print(f"nema_agg after dedup  : {len(nema_agg)} rows")


nema_agg after dedup  : 47 rows


In [14]:


# ── 1.10  Build master analytical frame ──────────────────────────────────

master = hh.copy()

# HH-level joins
master = master.merge(ind_agg, on='interview__key', how='left', suffixes=('', '_i'))
master = master.merge(dw_agg,  on='interview__key', how='left', suffixes=('', '_d'))
master = master.merge(lp_agg,  on='interview__key', how='left', suffixes=('', '_l'))

# County-level joins
master['a01'] = master['a01'].astype('Int64')
for cdf in [county_agg, nema_agg, wsvc_agg, mort_agg, loan_agg, fin_agg]:
    cdf['a01'] = cdf['a01'].astype('Int64')
    master = master.merge(cdf, on='a01', how='left')

# ── assertions ────────────────────────────────────────────────────────────
assert len(master) == len(hh), \
    f"Row inflation detected: {len(master):,} != {len(hh):,}"
assert master['interview__key'].nunique() == len(master), \
    "Duplicate interview__key values in master frame"

# ── summary ───────────────────────────────────────────────────────────────
added_cols = [c for c in master.columns if c not in hh.columns]

print()
print("=" * 60)
print("MASTER FRAME — BUILD COMPLETE")
print("=" * 60)
print(f"  Rows    : {len(master):,}  (spine intact)")
print(f"  Columns : {master.shape[1]}  ({len(hh.columns)} base + {len(added_cols)} added)")

layers = [
    ('household (base)',    hh.columns.tolist()),
    ('individual (agg)',    [c for c in ind_agg.columns  if c != 'interview__key']),
    ('dwelling (agg)',      [c for c in dw_agg.columns   if c != 'interview__key']),
    ('land_parcels (agg)',  [c for c in lp_agg.columns   if c != 'interview__key']),
    ('county_agg',          [c for c in county_agg.columns if c != 'a01']),
    ('nema_agg',            [c for c in nema_agg.columns   if c != 'a01']),
    ('wsvc_agg',            [c for c in wsvc_agg.columns   if c != 'a01']),
    ('mort_agg',            [c for c in mort_agg.columns   if c != 'a01']),
    ('loan_agg',            [c for c in loan_agg.columns   if c != 'a01']),
    ('fin_agg',             [c for c in fin_agg.columns    if c != 'a01']),
]
print(f"\n  {'Layer':<25} {'Cols':>6}")
print("  " + "─" * 33)
for name, cols in layers:
    print(f"  {name:<25} {len(cols):>6}")

high_miss = (
    master[added_cols].isna().mean()
    .pipe(lambda s: s[s > 0.50])
    .sort_values(ascending=False)
)
print()
if len(high_miss):
    print(f"  Added columns >50% missing ({len(high_miss)}):")
    for col, rate in high_miss.head(10).items():
        print(f"    {col:<42} {rate*100:.0f}%")
    if len(high_miss) > 10:
        print(f"    ... and {len(high_miss)-10} more")
else:
    print("  No added columns exceed 50% missing.")

print(f"\n  master is ready.  Shape: {master.shape}")


MASTER FRAME — BUILD COMPLETE
  Rows    : 21,347  (spine intact)
  Columns : 443  (392 base + 51 added)

  Layer                       Cols
  ─────────────────────────────────
  household (base)             392
  individual (agg)              10
  dwelling (agg)                11
  land_parcels (agg)             7
  county_agg                     5
  nema_agg                       3
  wsvc_agg                       5
  mort_agg                       4
  loan_agg                       3
  fin_agg                        3

  Added columns >50% missing (11):
    mort_interest_rate                         81%
    fin_avg_tenure_months                      64%
    mort_ltv_ratio                             64%
    mort_avg_term_years                        60%
    lp_any_collateral                          56%
    lp_any_dispute                             56%
    lp_any_registered                          56%
    lp_primary_use                             56%
    lp_has_title             

In [15]:
master.head()

,interview__key,interview__id,a01,countycode,a07_1,serial,a12,c01_1,c01_1other,c01_2,c01_2other,c01_3,c01_4,c01_5,c02_1,c02_1other,c02_2,c02_2other,c02_3,c02_4,...,cty_has_housing_policy,cty_building_approval_system,nema_eia_applications,nema_eia_approvals,nema_processing_days,wsvc_water_connections,wsvc_sewer_connections,wsvc_water_tariff,wsvc_service_quality,wsvc_n_providers,mort_interest_rate,mort_ltv_ratio,mort_avg_term_years,mort_n_providers,loan_avg_size,loan_outstanding,loan_n_providers,fin_portfolio,fin_avg_tenure_months,fin_n_financiers
0,00-00-55-14,8a585d4dd71641b8a1348be6cc11e121,31,31,2,35312800,6,1,,2,,0,NaN,NaN,1,,2,,0,NaN,...,NaN,1.0000,17.0000,42.0000,30.0000,1592.0000,1231.0000,5000.0000,1.0000,2.0000,1.5333,18.6475,1.8000,3.0000,285184303.9000,647953187.0000,10.0000,398702274.0000,13.0000,2.0000
1,00-01-22-52,d155c88b64de40148cda8dd079c36baa,14,14,2,44136136,3,1,,1,,0,NaN,NaN,1,,1,,0,NaN,...,NaN,NaN,169.0000,151.0000,5.0000,4491.0000,4166.0000,1000.0000,1.4000,5.0000,1.0000,100.0000,1.0000,4.0000,239184563.1246,1779292.7007,13.0000,13775733071.7700,15.0000,4.0000
2,00-02-11-67,b511ae4612704d9b9d395c901d975644,38,38,2,99694240,7,7,,5,,0,30.0000,1.0000,7,,5,,0,30.0000,...,0.0000,1.0000,158.0000,123.0000,13.0000,0.0000,251.0000,0.0000,2.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00-02-70-91,e1ff2d92960947d8adc30f55ff666b15,7,07,1,92351040,6,6,,5,,1,20.0000,1.0000,6,,5,,1,20.0000,...,NaN,3.0000,2.0000,5.0000,21.0000,300.0000,382.0000,3000.0000,1.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00-02-78-46,522414c7575543ddb88782491aa5e190,14,14,2,26181404,4,1,,1,,0,NaN,NaN,1,,1,,0,NaN,...,NaN,NaN,169.0000,151.0000,5.0000,4491.0000,4166.0000,1000.0000,1.4000,5.0000,1.0000,100.0000,1.0000,4.0000,239184563.1246,1779292.7007,13.0000,13775733071.7700,15.0000,4.0000


In [16]:
#null count as percentage

master_null = master.isnull()
master_null_percent = master_null.mean()
master_null_percent


,0
interview__key,0.0000
interview__id,0.0000
a01,0.0000
countycode,0.0000
a07_1,0.0000
...,...
loan_outstanding,0.2051
loan_n_providers,0.1861
fin_portfolio,0.1861
fin_avg_tenure_months,0.6404


In [17]:
# Save it
master.to_parquet(PQ / 'master_frame.parquet')